**BD DE SOLO 2026**

In [1]:
import pandas as pd
import os
from datetime import datetime
from tqdm import tqdm  # Biblioteca para mostrar barra de progreso

# Función para combinar cabecera y datos
def combinar_cabecera_y_datos(columna, primera_fila):
    if pd.isna(primera_fila):
        return columna
    return primera_fila

# Ruta base del archivo (modifica según tu computadora)
base = r'C:\Users\varce\Macroconsult S.A'

# Fecha de corte para procesar archivos
fecha_corte = datetime.strptime('2024-12-18', '%Y-%m-%d')

# DataFrame para concatenar resultados
df_BD_Supply_IEOD = pd.DataFrame()

# Directorio de las BBDD
for A_o in list(range(2026, 2027)):  # Definir periodo
    ruta = r'Soporte TI - REGCOM\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD'
    Base_ruta = os.path.join(base, ruta)
    A_o = str(A_o)
    ruta_IEOD = os.path.join(Base_ruta, A_o)

    # Listar archivos y filtrar solo los posteriores a la fecha de corte
    archivos_excel = [
        archivo for archivo in os.listdir(ruta_IEOD)
        if (archivo.endswith('.xlsx') or archivo.endswith('.xls')) and
           pd.to_datetime(archivo[:-5], format='%Y-%m-%d') > fecha_corte
    ]

    for archivo_excel in tqdm(archivos_excel, desc="Procesando archivos"):  # Barra de progreso
        ruta_archivo_excel = os.path.join(ruta_IEOD, archivo_excel)

        # Leer el archivo Excel
        with pd.ExcelFile(ruta_archivo_excel) as xls:

            # Buscar hoja con "DESPACHO_EJECUTADO" en el nombre
            hoja_seleccionada = [hoja for hoja in xls.sheet_names if "DESPACHO_EJECUTADO" in hoja]

            if hoja_seleccionada:  # Si se encuentra una hoja válida
                hoja = hoja_seleccionada[0]
                df_hoja = pd.read_excel(
                    ruta_archivo_excel,
                    sheet_name=hoja,
                    skiprows=4,
                    nrows=53
                )

                # Incorporar los cambios del código unitario:

                # 1. Eliminar la primera columna
                df_hoja = df_hoja.iloc[:, 1:]

                # 2. Rellenar espacios vacíos en la columna "HORA" con "HORA"
                if 'HORA' in df_hoja.columns:
                    df_hoja['HORA'] = df_hoja['HORA'].fillna('HORA')
                else:
                    # Si no existe una columna llamada "HORA", usa la primera columna
                    df_hoja.iloc[:, 0] = df_hoja.iloc[:, 0].fillna('HORA')

                # 3. Seleccionar la cabecera correcta
                df_cabecera_combinada = df_hoja.apply(lambda col: combinar_cabecera_y_datos(col.name, col.iloc[0]), axis=0)
                df_hoja = pd.concat([df_cabecera_combinada.to_frame().T, df_hoja.iloc[1:]])

                # 4. Seleccionar solo las filas desde la cuarta fila y tomar 49 filas
                df_hoja = df_hoja.iloc[4:]

                # 5. Reemplazar la cabecera con la primera fila
                df_hoja.columns = df_hoja.iloc[0]
                df_hoja = df_hoja[1:]

                # Eliminar columnas "Unnamed" y "MW"
                df_hoja = df_hoja.filter(regex='^(?!.*Unnamed).*$', axis=1)
                df_hoja = df_hoja.filter(regex='^(?!.*MW).*$', axis=1)

                # Reemplazar tab (\t) en los nombres de las columnas por un guion bajo (_)
                df_hoja.columns = df_hoja.columns.str.replace(r'\t', '_', regex=True)
          

                # Eliminar columnas completamente vacías
                df_hoja = df_hoja.dropna(axis=1, how='all')

                # Agregar columna de fecha basada en el nombre del archivo
                fecha = pd.to_datetime(archivo_excel[:-5], format='%Y-%m-%d')
                df_hoja['Fecha'] = fecha

                # Crear columna 'id' combinando 'Fecha' y 'HORA'
                if 'HORA' in df_hoja.columns:
                    df_hoja['id'] = df_hoja['Fecha'].astype(str) + '_' + df_hoja['HORA'].astype(str)

                # Concatenar al DataFrame final
                df_BD_Supply_IEOD = pd.concat([df_BD_Supply_IEOD, df_hoja], ignore_index=True)
   

Procesando archivos: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 98/98 [01:31<00:00,  1.07it/s]


In [2]:
df_BD_Supply_IEOD 
# Crear la columna 'id' combinando 'Fecha' y 'HORA'
df_BD_Supply_IEOD['id'] = df_BD_Supply_IEOD['Fecha'].astype(str) + '_' + df_BD_Supply_IEOD['HORA'].astype(str)

# Crear una copia desfragmentada del DataFrame
df_BD_Supply_IEOD = df_BD_Supply_IEOD.copy()

substring = 'CUPISNIQUE'
columnas_filtradas = df_BD_Supply_IEOD.filter(like=substring).columns
print(df_BD_Supply_IEOD[columnas_filtradas])

archivo = r'Soporte TI - REGCOM\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción_26.txt'
ruta_archivo = os.path.join(base, archivo)
df_BD_Supply_IEOD.to_csv(ruta_archivo, sep="|", index=False)

4    PQE-EOLICO-CUPISNIQUE
0                     23.3
1                     24.7
2                     24.4
3                     28.5
4                     20.7
...                    ...
4699                  32.9
4700                    28
4701                  23.9
4702                  22.4
4703                    19

[4704 rows x 1 columns]


###### **Actualización de la BD gral con la BD 2026**

In [3]:
# Actualizar la BD gral con la BD 2024

#Código para crear una bd de solo 2024.

import os
import pandas as pd
from openpyxl import load_workbook
import locale
from datetime import datetime
from tqdm import tqdm
import pandas as pd

# Cargar el nuevo conjunto de datos
# Ajusta la ruta del archivo y el nombre según sea necesario

base = r'C:\Users\varce\Macroconsult S.A' #Modificar a corde a tu compu 

archivo_act = r'Soporte TI - REGCOM\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción_26.txt'
ruta_archivo_act = os.path.join(base, archivo_act)
df_nuevo = pd.read_csv(ruta_archivo_act,sep='|')

archivo = r'Soporte TI - REGCOM\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción.txt'
ruta_archivo = os.path.join(base, archivo)
df_BD_Supply_IEOD = pd.read_csv(ruta_archivo,sep='|')

# Identificar registros únicos basándose en la columna 'id' o cualquier otra columna clave
# Este paso elimina los registros en df_nuevo que ya están presentes en df_bd
df_nuevo_limpio = df_nuevo[~df_nuevo['id'].isin(df_BD_Supply_IEOD['id'])]

# Concatenar el DataFrame limpio con el DataFrame de la base de datos existente
df_actualizado = pd.concat([df_BD_Supply_IEOD, df_nuevo_limpio], ignore_index=True)

# Opcional: guardar el DataFrame actualizado como un nuevo archivo Excel
archivo_act_ = r'Soporte TI - REGCOM\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción_Act.txt'
ruta_archivo_act_ = os.path.join(base, archivo_act_)
df_actualizado.to_csv(ruta_archivo_act_, sep="|", index=False)

C:\Users\varce\AppData\Local\Temp\ipykernel_14844\2432656832.py:24: DtypeWarning: Columns (131) have mixed types. Specify dtype option on import or set low_memory=False.
  df_BD_Supply_IEOD = pd.read_csv(ruta_archivo,sep='|')


**Otros códigos ad oc**

In [18]:
#Códigos Ad oc:

import os
import pandas as pd
from openpyxl import load_workbook
import locale
from datetime import datetime
from tqdm import tqdm

locale.setlocale(locale.LC_TIME, 'es_ES')


# Actualizar la BD

import pandas as pd

# Cargar el nuevo conjunto de datos
# Ajusta la ruta del archivo y el nombre según sea necesario

base = r'C:\Users\NB\Macroconsult S.A' #Modificar a corde a tu compu 

archivo_act = r'Regulación y Competencia - General\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción.txt'
ruta_archivo_act = os.path.join(base, archivo_act)
df_nuevo = pd.read_csv(ruta_archivo_act,sep='|')

df_nuevo.rename(columns={'8 DE AGOSTO': 'C.H.8 DE AGOSTO'}, inplace=True)

df_nuevo.columns = df_nuevo.columns.str.strip().str.replace('[\s\t\n]+', ' ', regex=True)

ruta_archivo_VF = r'C:\Users\NB\Macroconsult S.A\Regulación y Competencia - General\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD\IEOD - BBDD_Producción.txt'
df_nuevo.to_csv(ruta_archivo_VF, sep="|", index=False)


In [17]:
# Forma alternativa (Correr solo la primera)

import os
import pandas as pd
from openpyxl import load_workbook
import locale
from datetime import datetime
from datetime import timedelta
from tqdm import tqdm

locale.setlocale(locale.LC_TIME, 'es_ES')

def combinar_cabecera_y_datos(columna, primera_fila):
    if pd.isna(primera_fila):
        return columna
    return primera_fila
    
# Ruta al archivo de texto
ruta_archivo = r'C:\Users\egaray\Macroconsult S.A\Regulación y Competencia - General\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\IEOD - BBDD_Producción.txt' 

# Cargar el archivo de texto en un DataFrame
df_BD_Supply_IEOD = pd.read_csv(ruta_archivo, sep='|')

df_BD_Supply_IEOD['Fecha'] = pd.to_datetime(df_BD_Supply_IEOD['Fecha'], format='mixed')

# Identificar la última fecha en la columna
ultima_fecha = df_BD_Supply_IEOD['Fecha'].max()

# Generar una lista de fechas desde el día siguiente hasta el fin de año
fecha_siguiente = ultima_fecha + timedelta(days=1)
ayer = datetime.now() - timedelta(days=2)
lista_fechas = [fecha_siguiente + timedelta(days=d) for d in range((ayer - fecha_siguiente).days + 1)]

# Imprimir la última fecha y la lista de fechas
for fecha in tqdm(lista_fechas):
    A_o = str(fecha.year)
    Base_ruta = r'C:\Users\egaray\Macroconsult S.A\Regulación y Competencia - General\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\14 - IEOD'
    fecha_str = fecha.strftime('%Y-%m-%d')
    archivo = fecha.strftime('%Y-%m-%d') + '.xlsx'
    ruta_IEOD = os.path.join(Base_ruta, A_o, archivo)
    
    with pd.ExcelFile(ruta_IEOD) as xls:
        
        # Buscar hoja con "DESP" o "DESPACHO" en el nombre
        hoja_seleccionada = [hoja for hoja in xls.sheet_names if "DESPACHO_EJECUTADO" in hoja or "EJECUT" in hoja or "EJECUTADO" in hoja]

        # Leer la hoja y obtener las columnas
        df_hoja = pd.read_excel(xls, sheet_name = hoja_seleccionada, skiprows=4, nrows=49)
        
        # Identificar la clave del diccionario
        Clave = next(iter(df_hoja))

        # Quedarme con el contenido de la clave
        df_hoja = df_hoja[Clave]

        # Seleccionar la cabecera que sea correcta
        df_cabecera_combinada = df_hoja.apply(lambda col: combinar_cabecera_y_datos(col.name, col.iloc[0]), axis=0)
        df_hoja = pd.concat([df_cabecera_combinada.to_frame().T, df_hoja.iloc[1:]])
        
        # Reemplazar la cabecera por el contenido de la primera fila
        df_hoja.columns = df_hoja.iloc[0]
        
        # Eliminar la primera fila después de usarla como nueva cabecera
        df_hoja = df_hoja[1:]
         # Borrar todas las columnas que solo contengan NaN
        df_hoja = df_hoja.dropna(axis=1, how='all')

        # Borrar columnas unnamed y MW
        df_hoja = df_hoja.filter(regex='^(?!.*Unnamed).*$', axis=1)
        df_hoja = df_hoja.filter(regex='^(?!.*MW).*$', axis=1)

        # Identificar la fecha
        df_hoja['Fecha'] = pd.DataFrame({'Fecha': [fecha_str] * 49})
        
        df_hoja.columns = df_hoja.columns.str.strip().str.replace('[\s\t\n]+', ' ', regex=True)
        
        # Concatenar las bases una  bajo otra
        df_BD_Supply_IEOD = pd.concat([df_BD_Supply_IEOD, df_hoja], ignore_index=True)
        
ruta_archivo_VF = r'C:\Users\egaray\Macroconsult S.A\Regulación y Competencia - General\0_DATA\0.1_ELECTRICIDAD\0_BASES DE DATOS\COES\IEOD - BBDD_Producción_.txt'
df_BD_Supply_IEOD.to_csv(ruta_archivo_VF, sep="|", index=False)

0it [00:00, ?it/s]


In [1]:
import sys
print(sys.executable) 


c:\program files\python39\python.exe
